# 02 — Modeling: K-Means & DBSCAN
**Project 15 — Network Traffic Anomaly Detection (Clustering)**  
SUPMTI Rabat | AI Capstone 2025–2026 | Pr. Soufiane HAMIDA

This notebook justifies algorithm choices and hyperparameter selection.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from src.data.loader import NSL_KDD_COLUMNS
from src.data.preprocessor import fit_and_transform, save_processed
from src.evaluation.metrics import evaluate_clustering, cluster_label_table, identify_anomaly_cluster, error_analysis

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'KDDTrain+.txt'
TEST_PATH  = PROJECT_ROOT / 'data' / 'raw' / 'KDDTest+.txt'

train_df = pd.read_csv(TRAIN_PATH, names=NSL_KDD_COLUMNS)
test_df  = pd.read_csv(TEST_PATH,  names=NSL_KDD_COLUMNS)
print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')

## 1. Preprocessing

In [ ]:
X_train, preprocessor, y_train = fit_and_transform(train_df)
from src.data.preprocessor import transform_with_existing
X_test  = transform_with_existing(test_df, preprocessor)
y_test  = test_df['label'].copy()

# Save to data/processed/ for reproducibility
save_processed(X_train, y_train, preprocessor, split='train')
save_processed(X_test,  y_test,  preprocessor, split='test')

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

## 2. K-Means — Elbow Method
We sweep k from 2 to 12 and record inertia and silhouette score to choose the best k.

In [ ]:
# Use a sample for speed (full train set is 125k rows)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train), size=20000, replace=False)
X_sample = X_train[sample_idx]
y_sample = y_train.iloc[sample_idx]

k_range = range(2, 13)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sample)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sample, labels, sample_size=5000, random_state=42))
    print(f'k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil_scores[-1]:.4f}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers', name='Inertia'))
fig.update_layout(title='K-Means Elbow Curve', xaxis_title='k', yaxis_title='Inertia')
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(k_range), y=sil_scores, mode='lines+markers',
                          name='Silhouette', line=dict(color='orange')))
fig2.update_layout(title='Silhouette Score vs k', xaxis_title='k', yaxis_title='Silhouette Score')
fig2.show()

## 3. K-Means Final Model (k=5)
k=5 was selected based on the elbow curve inflection point and highest silhouette score.

In [ ]:
import joblib

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
train_clusters = kmeans.fit_predict(X_train)
test_clusters  = kmeans.predict(X_test)

# Save
joblib.dump(kmeans, PROJECT_ROOT / 'models' / 'kmeans_model.pkl')
joblib.dump(preprocessor, PROJECT_ROOT / 'models' / 'preprocessor.pkl')
print('Models saved.')

train_metrics = evaluate_clustering(X_train, train_clusters, y_train)
test_metrics  = evaluate_clustering(X_test,  test_clusters,  y_test)
print('\nTrain metrics:', train_metrics)
print('Test  metrics:', test_metrics)

In [ ]:
binary_table, original_table = cluster_label_table(test_clusters, y_test)
anomaly_id = identify_anomaly_cluster(binary_table)
print(f'Anomaly cluster identified: {anomaly_id}')
print('\nBinary table (test):')
display(binary_table)
print('\nAttack-type breakdown (test):')
display(original_table)

## 4. K-Means Error Analysis

In [ ]:
from src.models.kmeans_model import get_anomaly_mask
anomaly_mask_km = get_anomaly_mask(test_clusters, anomaly_id)

err = error_analysis(test_clusters, y_test, anomaly_mask_km)
print(f"Precision : {err['precision']:.4f}")
print(f"Recall    : {err['recall']:.4f}")
print(f"F1 Score  : {err['f1_score']:.4f}")
print(f"\nFalse Negatives (missed attacks): {err['false_negatives']}")
print('\nMissed attack types:')
print(err['missed_attack_types'].head(10))

## 5. DBSCAN — k-Distance Graph for eps Selection
We compute the distance to the k-th nearest neighbor for each point, sort it, and look for the 'knee' — this gives a good eps value.

In [ ]:
# Use a small sample — DBSCAN NN graph is O(n²)
rng2 = np.random.default_rng(0)
db_sample_idx = rng2.choice(len(X_test), size=3000, replace=False)
X_db_sample = X_test[db_sample_idx]

MIN_SAMPLES = 10
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_db_sample)
distances, _ = nbrs.kneighbors(X_db_sample)
k_distances = np.sort(distances[:, -1])

fig = go.Figure()
fig.add_trace(go.Scatter(y=k_distances, mode='lines', name='k-distance'))
fig.add_hline(y=2.0, line_dash='dash', line_color='red',
              annotation_text='eps=2.0 (chosen)', annotation_position='top left')
fig.update_layout(
    title=f'k-Distance Graph (k={MIN_SAMPLES}, sample={len(X_db_sample)})',
    xaxis_title='Points sorted by distance',
    yaxis_title=f'{MIN_SAMPLES}-NN distance',
)
fig.show()

## 6. DBSCAN Final Model (eps=2.0, min_samples=10)

In [ ]:
from src.models.dbscan_model import train_dbscan, get_dbscan_clusters, get_anomaly_mask_dbscan, get_dbscan_summary

dbscan = train_dbscan(X_test, eps=2.0, min_samples=10)
db_clusters = get_dbscan_clusters(dbscan)
db_summary  = get_dbscan_summary(db_clusters)
print('DBSCAN summary:', db_summary)

db_mask = get_anomaly_mask_dbscan(db_clusters)
db_metrics = evaluate_clustering(X_test, db_clusters, y_test)
print('DBSCAN metrics:', db_metrics)

In [ ]:
db_err = error_analysis(db_clusters, y_test, db_mask)
print(f"DBSCAN Precision : {db_err['precision']:.4f}")
print(f"DBSCAN Recall    : {db_err['recall']:.4f}")
print(f"DBSCAN F1 Score  : {db_err['f1_score']:.4f}")

## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'K-Means (k=5)',
        'Silhouette': test_metrics['silhouette_score'],
        'Davies-Bouldin': test_metrics['davies_bouldin_score'],
        'Adj. Rand Index': test_metrics['adjusted_rand_index'],
        'Precision': err['precision'],
        'Recall': err['recall'],
        'F1 Score': err['f1_score'],
    },
    {
        'Model': 'DBSCAN (eps=2.0)',
        'Silhouette': db_metrics['silhouette_score'],
        'Davies-Bouldin': db_metrics['davies_bouldin_score'],
        'Adj. Rand Index': db_metrics['adjusted_rand_index'],
        'Precision': db_err['precision'],
        'Recall': db_err['recall'],
        'F1 Score': db_err['f1_score'],
    },
])
display(comparison.set_index('Model'))